# CS2 (국내반) --- Lecture 10: Poker Hands
### Problem Set (in-class)

*이 노트북은 채점되지 않습니다. 수업 시간에 바로 이어서 풀면 됩니다.*

#### 먼저 실행: `Card` class와 도우미 함수

아래 셀은 **수정하지 않고 맨 먼저 실행**합니다. (`Hand` class는 Problem 1 셀에 있습니다.)
`mk((suit, rank), ...)`는 확인용 `Hand`를 쉽게 만들어주는 도우미 함수입니다.

In [ ]:
import random

class Card:
    def __init__(self, suit, rank):
        self.suit = suit    # 0,1,2,3  (♣,◇,♡,♠)
        self.rank = rank    # 1~13     (A,2,...,10,J,Q,K)

    def __repr__(self):     # print(card)했을 때 보기 좋게
        return "♣◇♡♠"[self.suit] + ["A", "2", "3", "4", "5", "6", "7", "8", "9",
                                     "10", "J", "Q", "K"][self.rank - 1]

def mk(*pairs):
    """mk((0, 5), (1, 13)) -> Hand([Card(0, 5), Card(1, 13)])"""
    return Hand([Card(s, r) for s, r in pairs])

### Problem 1

Write a function `getNumCardsBySuit` that work on the class `Hand`:
- input parameter: `self`
- return value: list `self.cards`에 포함된 5개의 `Card` object들의
  suit값(0,1,2,3)의 등장 횟수를 기록한 새로운 list
  - 수업 슬라이드에서 세로축으로 그려진 list `numCardsBySuit`로 ♣ 모양의
    카드의 갯수는 `numCardsBySuit[0]`, ..., ♠ 모양의 카드의 갯수는
    `numCardsBySuit[3]`
  - `numCardsBySuit`의 index는 각 `Card` object의 suit 값에 대응됨

In [ ]:
class Hand:
    def __init__(self, cards):
        self.cards = cards   # Card object들의 list

    def getNumCardsByRank(self):    # 이미 주어진 메서드 (참고용)
        numCardsByRank = [0] * 13
        for i in range(len(self.cards)):
            j = self.cards[i].rank
            assert 1 <= j <= 13
            numCardsByRank[j - 1] += 1
        return numCardsByRank

    def getNumCardsBySuit(self):    # <- Problem 1: 이 메서드를 완성
        numCardsBySuit = [0] * 4
        # ADD ADDITIONAL CODE HERE!!
        # (getNumCardsByRank를 참고: rank 대신 suit를 세면 됨. index는 0~3)
        return numCardsBySuit

In [ ]:
# 문제 1 확인용 (위 셀을 먼저 실행)
h = mk((0, 1), (0, 5), (1, 3), (3, 3), (3, 9))
print(h.cards)                  # [♣A, ♣5, ◇3, ♠3, ♠9]
print(h.getNumCardsBySuit())    # [2, 1, 0, 2]
print(mk((2, 1), (2, 4), (2, 6), (2, 9), (2, 12)).getNumCardsBySuit())  # [0, 0, 5, 0]

#### 확인용 예시 hand 모음 (실행만 하면 됨)

Problem 2~4의 확인용으로 쓰이는 hand들(`samples`)과, 함수가 맞게 동작하는지 한 번에 점검해주는 `check` 함수입니다.
`check("이름", 함수, 정답인_족보들...)`는 모든 예시 hand에 대해 함수 결과를 확인하고 틀린 것만 알려줍니다.

In [ ]:
samples = [
    ("straightFlush", mk((0, 5), (0, 6), (0, 7), (0, 8), (0, 9))),
    ("straightFlush", mk((1, 10), (1, 11), (1, 12), (1, 13), (1, 1))),   # 10-J-Q-K-A
    ("quadruple",     mk((0, 9), (1, 9), (2, 9), (3, 9), (0, 2))),
    ("fullHouse",     mk((0, 3), (1, 3), (2, 3), (0, 8), (1, 8))),
    ("flush",         mk((2, 1), (2, 4), (2, 6), (2, 9), (2, 12))),
    ("straight",      mk((0, 4), (1, 5), (2, 6), (3, 7), (0, 8))),
    ("triple",        mk((0, 7), (1, 7), (2, 7), (0, 2), (1, 11))),
    ("twoPair",       mk((0, 6), (1, 6), (0, 10), (1, 10), (2, 3))),
    ("pair",          mk((0, 6), (1, 6), (0, 10), (1, 12), (2, 3))),
    ("highCard",      mk((0, 1), (1, 3), (2, 5), (3, 7), (0, 9))),
]

def check(label, f, *trueCategories):
    """samples 중 족보가 trueCategories에 속하는 hand에서만 f가 True여야 함"""
    ok = True
    for category, hand in samples:
        got, want = f(hand), (category in trueCategories)
        if got != want:
            ok = False
            print("  X", label, "on", category, hand.cards, "-> got", got, ", expected", want)
    print("OK " if ok else "NO ", label)

### Problem 2

Write functions for determining "straight flush"/"straight"/"flush":

1. Write a function `atLeastStraight`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "straight" or "straight flush"),
     `False`(otherwise)
   - *suit가 모두 같은지 여부는 따질 필요 없고 rank 값만 따져보면 되므로
     `L = hand.getNumCardsByRank()`로 얻은 `L` 정보만으로 충분. 주어진
     함수 `hasConsecutivePositive`를 이용하면 코드가 매우 간단해짐.*
2. Write a function `atLeastFlush`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "flush" or "straight flush"),
     `False`(otherwise)
   - *rank 정보는 필요없고 suit 값만 따져보면 되므로 `L =
     hand.getNumCardsBySuit()`로 얻은 `L` 정보만으로 충분*
3. Write a function `straightFlush`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "straight flush"), `False`(otherwise)
   - *함수 `atLeastStraight`와 `atLeastFlush`를 이용하면 매우 간단하게
     해결됨*
4. Write a function `flush`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "flush"), `False`(otherwise)
   - *tie-breaking rule에 의해 "straight flush" 경우는 제외해줘야 함. 위의
     함수들을 이용하면 쉽게 해결됨*
5. Write a function `straight`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "straight"), `False`(otherwise)
   - *마찬가지로 "straight flush" 경우는 제외해줘야 함*

In [ ]:
# use this function for atLeastStraight()
def hasConsecutivePositive(L, k):
    n = len(L)
    assert k <= n
    counter = 0
    for i in range(n + 1):  # +1: account for wrapping
        if L[i % n] >= 1:
            counter += 1
            if counter == k:
                return True
        else:
            counter = 0  # reset
    return False

def atLeastStraight(hand):
    L = hand.getNumCardsByRank()
    # ADD ADDITIONAL CODE HERE!!
    # (L에서 양수가 5개 연속인지 -> 주어진 hasConsecutivePositive 이용)
    pass

def atLeastFlush(hand):  # for some pattern
    L = hand.getNumCardsBySuit()
    # ADD ADDITIONAL CODE HERE!!
    # (L의 원소 중 하나라도 5인가?  "for some" 패턴: for문 + return True, 끝나면 return False)
    pass

def straightFlush(hand):
    # ADD ADDITIONAL CODE HERE!!
    # (atLeastStraight(hand)와 atLeastFlush(hand)를 이용)
    pass

def flush(hand):
    # ADD ADDITIONAL CODE HERE!!
    # (atLeastFlush이지만 straightFlush는 아닌 경우)
    pass

def straight(hand):
    # ADD ADDITIONAL CODE HERE!!
    # (atLeastStraight이지만 straightFlush는 아닌 경우)
    pass

In [ ]:
# 문제 2 확인용 (위 셀들을 먼저 실행)  -- 틀린 경우에만 X 줄이 나옵니다
check("atLeastStraight", atLeastStraight, "straightFlush", "straight")
check("atLeastFlush",    atLeastFlush,    "straightFlush", "flush")
check("straightFlush",   straightFlush,   "straightFlush")
check("flush",           flush,           "flush")
check("straight",        straight,        "straight")

### Problem 3

Write functions for determining "quadruple"/"full house"/"triple"/
"two pair"/"pair":

1. Write a function `hasMultipleCardsWithSameRank`:
   - input parameter: a `Hand` object `hand` and an integer `num`
   - return value: `True`(if hand contains **exactly** `num` cards of a
     certain rank), `False`(otherwise)
   - *예를 들어, `hasMultipleCardsWithSameRank(hand,4)`는 `hand`가
     "quadruple"일 때 `True`이며 그 역도 성립한다.*
2. Write a function `fullHouse`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "full house"), `False`(otherwise)
   - *`hasMultipleCardsWithSameRank`를 이용하면 간단하게 해결됨*
3. Write a function `triple`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "triple"), `False`(otherwise)
   - *함수 `hasMultipleCardsWithSameRank`를 이용하면 간단하게 해결됨*
   - *tie-breaking rule에 의해 "full house"와 "quadruple" 경우는
     제외해줘야 함*
4. Write a function `twoPair`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "two pair"), `False`(otherwise)
   - *`hasMultipleCardsWithSameRank`는 "for some" 패턴으로 만들어졌는데,
     이와 유사하게 counter 패턴으로 변형*
   - *tie-breaking rule에 의해 "full house" 경우는 제외해야*
5. Write a function `pair`:
   - input parameter: a `Hand` object `hand`
   - return value: `True`(if hand is "pair"), `False`(otherwise)
   - *`hasMultipleCardsWithSameRank`를 이용*
   - *tie-breaking rule에 의해 "full house", "two pair", "quadruple",
     "triple" 경우는 제외해야*

In [ ]:
def hasMultipleCardsWithSameRank(hand, num):  # for some pattern
    L = hand.getNumCardsByRank()
    # ADD ADDITIONAL CODE HERE!!
    # (L의 원소 중 정확히 num인 것이 하나라도 있는가?)
    for i in range(len(L)):
        pass
    return False

def quadruple(hand):
    return hasMultipleCardsWithSameRank(hand, 4)

def fullHouse(hand):
    # ADD ADDITIONAL CODE HERE!!
    # (같은 rank 3장 + 같은 rank 2장)
    pass

def triple(hand):
    # ADD ADDITIONAL CODE HERE!!
    # (3장짜리는 있지만 full house는 아닌 경우. quadruple은 3장짜리가 없으므로 자동 제외)
    pass

def twoPair(hand):  # counter pattern
    L = hand.getNumCardsByRank()
    # ADD ADDITIONAL CODE HERE!!
    # (L에서 정확히 2인 원소의 개수를 세어 counter에 저장)
    counter = 0
    for i in range(len(L)):
        pass
    return False   # counter를 이용해 고치기. full house 제외 주의!

def pair(hand):
    # ADD ADDITIONAL CODE HERE!!
    # (2장짜리가 있지만 full house, two pair는 아닌 경우)
    pass

In [ ]:
# 문제 3 확인용 (위 셀을 먼저 실행)  -- 틀린 경우에만 X 줄이 나옵니다
check("hasMultipleCardsWithSameRank(hand, 4)", lambda h: hasMultipleCardsWithSameRank(h, 4), "quadruple")
check("hasMultipleCardsWithSameRank(hand, 3)", lambda h: hasMultipleCardsWithSameRank(h, 3), "fullHouse", "triple")
check("hasMultipleCardsWithSameRank(hand, 2)", lambda h: hasMultipleCardsWithSameRank(h, 2), "fullHouse", "twoPair", "pair")
check("quadruple", quadruple, "quadruple")
check("fullHouse", fullHouse, "fullHouse")
check("triple",    triple,    "triple")
check("twoPair",   twoPair,   "twoPair")
check("pair",      pair,      "pair")

### Problem 4

Write a function `countPokerHands`:
- input parameter: a list of `Hand` objects (함수 `randomHand`로
  랜덤하게 생성한 수십만~수천만 개의 `Hand` object들의 list)
- return value: `Hand` object들을 "straight flush", "quadruple", "full
  house", "flush", "straight", "triple", "two pair", "pair"로
  분류했을때 각 포커패에 속하는 것들의 갯수를 세어서 이 순서대로 나열한
  list

In [ ]:
# counter pattern
def countPokerHands(hands):
    nStraightFlush = 0
    nQuadruple = 0
    nFullHouse = 0
    nFlush = 0
    nStraight = 0
    nTriple = 0
    nTwoPair = 0
    nPair = 0

    for i in range(len(hands)):
        hand = hands[i]
        # 순위가 높은 것부터 검사하고, 하나에 걸리면 나머지는 검사하지 않기 -> if / elif 사슬
        # False 자리를 알맞은 함수 호출로 바꾸기 (예: straightFlush(hand))
        if False:
            nStraightFlush += 1
        elif False:
            nQuadruple += 1
        elif False:
            nFullHouse += 1
        elif False:
            nFlush += 1
        elif False:
            nStraight += 1
        elif False:
            nTriple += 1
        elif False:
            nTwoPair += 1
        elif False:
            nPair += 1

    return [nStraightFlush, nQuadruple, nFullHouse, nFlush,
            nStraight, nTriple, nTwoPair, nPair]

In [ ]:
# 문제 4 확인용 (Problem 1~4 셀을 먼저 실행): samples의 10개 hand를 넣어보면
print(countPokerHands([hand for category, hand in samples]))
# [2, 1, 1, 1, 1, 1, 1, 1]   (straight flush 2개, 나머지 1개씩; high card 1개는 어디에도 안 셈)

아래 셀은 5장짜리 hand를 무작위로 10만 개 만들어 확률(%)을 추정합니다 (Monte Carlo simulation).
이론값은 대략 straight flush 0.0015, quadruple 0.024, full house 0.144, flush 0.197, straight 0.392,
triple 2.11, two pair 4.75, pair 42.26 입니다. (`nTrials`가 작으면 희귀한 족보는 오차가 큽니다.)

In [ ]:
def randomHand(numCards):
    cards = []
    for suit in range(4):
        for rank in range(1, 13+1):
            cards.append(Card(suit, rank))
    random.shuffle(cards)
    return Hand(cards[:numCards])

def test():
    print("\nWait for your program to estimate probabilities with 5-card hand...")

    hands = []
    nTrials = 100000  # 26000000
    for i in range(nTrials):
        hands.append(randomHand(5))
    L = countPokerHands(hands)

    print("Straight Flush:" + str(1.0*L[0]/nTrials*100))
    print("Quadruple:" + str(1.0*L[1]/nTrials*100))
    print("Full House:" + str(1.0*L[2]/nTrials*100))
    print("Flush:" + str(1.0*L[3]/nTrials*100))
    print("Straight:" + str(1.0*L[4]/nTrials*100))
    print("Triple:" + str(1.0*L[5]/nTrials*100))
    print("TwoPair:" + str(1.0*L[6]/nTrials*100))
    print("Pair:" + str(1.0*L[7]/nTrials*100))

test()

### Optional Problems (Problem 5~11)

필수 문제와 달리 제출/검사 대상은 아니지만, 큰 도움이 되므로 시간이 남으면 모두 시도해보는 것을 권합니다 (대부분 기출문제입니다).

### Problem 5

다음과 같이 정의된 함수 `nDistinct`를 완성하라:
- 입력: `Hand` object인 `hand`
  - `hand`에 포함된 카드의 갯수는 5가 아닐 수 있다
  - `hand`에 포함된 카드들 중 동일한 suit/rank를 가진 카드들도 있다
- 리턴값: `hand`에 포함된 카드들 중 다른 것들의 갯수

In [ ]:
def nDistinct(hand):
    # your code here
    pass

In [ ]:
# 문제 5 확인용
h = mk((0, 1), (0, 1), (1, 1), (2, 5), (2, 5), (2, 5))
print(nDistinct(h))   # 3   ((♣A), (◇A), (♡5) 세 종류)
print(nDistinct(mk((0, 1), (1, 2), (2, 3))))   # 3

### Problem 6

다음과 같이 정의된 함수 `k_flush`를 완성하라:
- 입력: 정수 `k >= 0`와 `Hand` object인 `hand`
  - `hand`에 포함된 카드의 갯수는 5가 아닐 수 있다
  - `hand`에 포함된 카드들 중 동일한 suit/rank를 가진 카드는 없다
- 리턴값: `hand`에 `ℓ`장의 카드를 추가해서 `k`-flush로 만들 수 있는 `ℓ`중
  최소값
  - `k`개 이상의 동일한 suit를 포함한 `hand`를 `k`-flush로 부른다

In [ ]:
def k_flush(k, hand):
    # your code here
    pass

In [ ]:
# 문제 6 확인용
h = mk((0, 1), (0, 2), (1, 3), (2, 4))   # ♣ 2장, ◇ 1장, ♡ 1장
print(k_flush(2, h))   # 0
print(k_flush(3, h))   # 1
print(k_flush(5, h))   # 3

### Problem 7

다음과 같이 정의된 함수 `k_straight`를 완성하라:
- 입력: 양의 정수 `k <= 13`과 `Hand` object인 `hand` (`hand`의 조건은
  6번 문제와 동일)
- 리턴값: `hand`에 `ℓ`장의 카드를 추가해서 `k`-straight로 만들 수 있는
  `ℓ`중 최소값
  - `k`개 이상의 consecutive rank를 포함한 `hand`를 `k`-straight로
    부른다
  - 이 문제에서의 `k`-straight는 모든 "wrap-around"를 허용한다. 예를
    들어, Jack-Queen-King-Ace, Queen-King-Ace-2, King-Ace-2-3는 모두
    4-straight이다. (수업시간에 다루었던 straight는 마지막이 Ace인
    경우까지만 허용)

In [ ]:
def k_straight(k, hand):
    # your code here
    # (힌트 없이 막히면: rank가 원형(1~13, 13 다음은 다시 1)으로 놓여 있다고 생각하고,
    #  길이 k짜리 "창"을 한 칸씩 돌려가며 hand에 이미 있는 rank가 몇 개인지 세어 보기)
    pass

In [ ]:
# 문제 7 확인용
print(k_straight(2, mk((0, 1), (1, 2), (2, 4), (3, 5))))    # 0  (A, 2)
print(k_straight(3, mk((0, 1), (1, 2), (2, 4), (3, 5))))    # 1  (2-3-4 에 3을 추가)
print(k_straight(5, mk((0, 1), (1, 2), (2, 4), (3, 5))))    # 1  (A-2-3-4-5 에 3을 추가)
print(k_straight(4, mk((0, 12), (1, 13), (2, 1))))          # 1  (J-Q-K-A 에 J를 추가)
print(k_straight(4, mk((0, 13), (1, 1), (2, 2), (3, 3))))   # 0  (K-A-2-3, wrap-around)

### Problem 8

다음과 같이 정의된 함수 `k_straight_flush`를 완성하라:
- 입력: 양의 정수 `k <= 13`과 `Hand` object인 `hand`
  - `hand`에 포함된 카드의 갯수는 5가 아닐 수 있다
  - `hand`에 포함된 카드들 중 동일한 suit/rank를 가진 카드는 없다
- 리턴값: `hand`에 `ℓ`(≥0)장의 카드를 추가해서 `k`-straight-flush로 만들
  수 있는 `ℓ`중 최소값
  - `hand`에 포함된 어떤 `k`개의 카드들이 존재하여, 이들이 모두 같은
    suit를 가지고 rank들이 "`k`-straight"하면 `hand`가
    `k`-straight-flush라고 부른다
  - 이 문제에서의 `k`-straight는 (7번 문제와 마찬가지로) 모든
    "wrap-around"를 허용한다.
  - *6, 7번 문제를 일반화시킨 것으로 `k`-straight-flush는
    `k`-straight와 `k`-flush를 단순히 and로 연결한 것이 아님에 유의.*

In [ ]:
def k_straight_flush(k, hand):
    # your code here
    pass

In [ ]:
# 문제 8 확인용
print(k_straight_flush(4, mk((0, 1), (0, 2), (0, 4), (1, 3), (1, 5))))   # 1  (♣A-2-3-4 에서 ♣3 추가)
print(k_straight_flush(2, mk((0, 1), (0, 2), (0, 4), (1, 3), (1, 5))))   # 0
# k-straight(0)와 k-flush(2)를 단순히 and로 연결하면 안 되는 예:
print(k_straight_flush(4, mk((0, 1), (0, 2), (1, 3), (1, 4))))           # 2

### Problem 9

포커 규칙을 다음과 같이 약간 변형한 상황을 고려하자: rank가 2인
카드는 임의의 suit/rank로 변환할 수 있다(변환하지 않아도 됨). 예를 들어,
hand `(♣2, ♡3, ♣3, ♠3, ♡4)`는 다음 중 임의로 변환해도 된다:
`(◇3, ♡3, ♣3, ♠3, ♡4)`(quadruple), `(♣4, ♡3, ♣3, ♠3, ♡4)`(full house),
`(♣2, ♡3, ♣3, ♠3, ♡4)`(triple, as it is without being changed).

다음과 같이 정의된 함수 `full_house`를 완성하라:
- 입력: 5개의 `Card`로 구성된 `Hand` object (11주차 문제와 같은 형태)
- 리턴값: `True`(주어진 입력을 위 규칙에 따라 변환하면 full house가 될
  수 있을 때), `False`(otherwise)
  - 예: 입력이 `(♣2, ♡3, ♣3, ♠3, ♡4)`면 `♣2`를 `♣4`로 변환하여 full
    house로 만들 수 있으므로 `True`를 리턴

In [ ]:
def full_house(hand):
    # your code here
    pass

In [ ]:
# 문제 9 확인용
print(full_house(mk((0, 2), (2, 3), (0, 3), (3, 3), (2, 4))))   # True   (♣2 -> ♣4)
print(full_house(mk((0, 5), (1, 5), (2, 7), (3, 9), (0, 11))))  # False
print(full_house(mk((0, 5), (1, 5), (2, 5), (3, 9), (0, 11))))  # False  (2가 없으니 변환 불가)
print(full_house(mk((0, 2), (1, 2), (2, 2), (3, 2), (0, 5))))   # True

### Problem 10

이 문제에서는 "#11: Poker Hands"에서 hand에 포함된 카드의
개수가 5가 아닌 경우를 고려한다(6개 이상의 카드들로 hand가 구성될 수도
있고, 4개 이하의 카드들로 구성될 수도 있다). 이러한 상황에서 straight,
flush, straight flush를 다음과 같이 재정의하자:
1. **straight**: 모든 카드들의 rank들을 연속되게 나열할 수 있을 경우
   - 7, 8번 문제에서와 같이 모든 "wrap-arounds"를 허용한다.
   - 예를 들어, 6장의 카드 `(♣King), (♠Ace), (♡2), (◇3), (♣4), (♣5)`로
     구성된 hand는 straight이다
2. **flush**: 모든 카드들이 같은 suit를 가지는 경우
3. **straight flush**: 위 두 조건을 모두 만족하는 경우

위 정의에 따르면 straight flush인 hand는 straight일 수도 있고 flush일
수도 있는데, 이러한 hand는 straight flush로만 분류하고 straight나
flush로는 분류하지 않는다. 즉:
- **straight**: 조건 1은 만족하되 조건 2는 만족하지 않는 경우
- **flush**: 조건 2는 만족하되 조건 1은 만족하지 않는 경우

로 분류한다.

다음과 같이 정의된 함수 `flush`를 완성하라:
- 입력: `Hand` object인 `hand`
  - `hand`에 포함된 카드의 개수는 3 이상 12 이하
  - `hand`에 포함된 카드들 중 동일한(suit와 rank가 모두 일치하는)
    카드는 없음
- 리턴값: `1`(hand가 flush인 경우), `0`(그렇지 않은 경우)
  - 상술한 바와 같이 hand가 straight flush이면 이는 flush로 분류하지
    않으므로 `0`을 리턴해야 한다.

In [ ]:
# 주의: Problem 2의 flush와 이름이 같습니다. 이 셀을 실행하면 Problem 2의 flush가 덮어써지므로,
# Problem 4를 다시 돌려야 한다면 Problem 2 셀을 다시 실행하세요.
def flush(hand):
    # your code here
    pass

In [ ]:
# 문제 10 확인용
print(flush(mk((0, 1), (0, 5), (0, 9))))                       # 1  (같은 suit, rank 연속 아님)
print(flush(mk((0, 5), (0, 6), (0, 7))))                       # 0  (straight flush)
print(flush(mk((0, 13), (0, 1), (0, 2))))                      # 0  (wrap-around straight flush)
print(flush(mk((0, 1), (1, 5), (0, 9))))                       # 0  (suit가 다름)
print(flush(mk((2, 2), (2, 4), (2, 6), (2, 8), (2, 10), (2, 12))))  # 1  (6장)

### Problem 11

주어진 hand에 포함된 카드들 중 세개를 적절히 선택하여 세개의
rank를 더한 값을 최대화하되 주어진 자연수 `k` 값은 넘지 않도록 하려고
한다. 즉, 세개의 rank를 더한 값들 중 `k` 이하인 것들만 골라내고 이들 중
최대값을 hand의 "`k`-점수"로 정의하자.

다음과 같이 정의된 함수 `k_score`를 완성하라:
- 입력: `Hand` object인 `hand`와 정수 `k`
  - `hand`에 포함된 카드의 갯수는 5가 아닐 수 있다.
  - `hand`에 포함된 카드들 중 suit와 rank 모두 동일한 카드는 없다.
    하지만 suit만 동일하거나 rank만 동일한 카드는 있을 수 있다.
- 리턴값: `hand`의 "`k`-점수"
  - 세개의 카드의 rank를 더한 값이 모두 `k`보다 크면 `0`을 리턴

In [ ]:
def k_score(hand, k):
    # your code here
    pass

In [ ]:
# 문제 11 확인용
h = mk((0, 3), (1, 5), (2, 8), (3, 9))
print(k_score(h, 17))   # 17  (3+5+9)
print(k_score(h, 16))   # 16  (3+5+8)
print(k_score(h, 10))   # 0   (어떤 세 장도 합이 10 이하가 안 됨)